# Figure SX: SO-EPT Coupling and SWCF - Historical vs. PiControl 

This analysis sanity checks that SO-EPT Coupling and SWCF calculate from piControl as similar to those derived from historical. This analysis is in response to a reviewer comment

In [16]:
import xarray as xr
import xcdat as xc
import numpy as np
import xskillscore as xscore
import matplotlib.pyplot as plt
import os
from numpy.polynomial.polynomial import polyfit, polyval

import os
import cartopy
import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xcdat as xc
import xskillscore as xscore

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import linregress
from typing import Tuple
from matplotlib.colors import to_rgba


# Ignore xarray warnings (bad practice)
import warnings
warnings.simplefilter("ignore") 

# Utils

In [11]:
def detrend(da):
    time_idx = xr.DataArray(np.arange(len(da.time), dtype="float32"), dims="time")
    slope = xscore.linslope(time_idx, da, dim="time", skipna=True)
    da = da - slope*time_idx
    return da 

def calculate_cf(tos: xr.Dataset, swcre: xr.Dataset, save: bool = False, save_name: str = "") -> xr.Dataset:
    """Calculate the shortwave cloud feedback (swcf)

    Definition 1:
    swcf = dR_swcre / dSST (Wm^-2/K)

    Definition 2:
    swcre_anoms = swcf * SST_anoms + b

    Args:
        tos (xr.Dataset): monthly sst anomalies
        swcre (xr.Dataset): monthly swcre anomalies
        save (bool, optional): Defaults to False.
        save_name (str, optional): Defaults to "".
Ahjhvi
    Returns:
        swcf (xr.Dataset): dimensions (model, time, lat, lon)
    """
    # Rechunk along time dimension
    tos = tos.chunk({'time': -1})
    swcre = swcre.chunk({'time': -1})

    tos = remove_land_full(tos, var="tos")
    swcre = remove_land_full(swcre, var="swcre")

    shared_models = list(set(tos.model.values).intersection(set(swcre.model.values)))

    # Calculate the correlation coefficient
    # dcre(x,y)/dsst(x,y)
    swcf = xscore.linslope(tos.sel(model=shared_models), swcre.sel(model=shared_models), dim='time', skipna=True)

    swcf_all = xr.Dataset({
        "swcf": swcf, 
    })
    return swcf_all

def fix_coords_no_time(data):
    data = data.bounds.add_bounds("X")
    data = data.bounds.add_bounds("Y")
    data = xc.swap_lon_axis(data, to=(-180, 180))
    return data

# Get shared models between swcf_eastsa_cmip6 and tos_grad_trend_cmip6
def get_shared_models(ds1, ds2):
    shared_models = list(set(ds1.model.values).intersection(set(ds2.model.values)))
    return ds1.sel(model=shared_models), ds2.sel(model=shared_models)

def remove_land_full(ds, var="skt"):
    ds = ds.rename(var).to_dataset()
    ds = xc.swap_lon_axis(ds, to=(-180, 180))
    from global_land_mask import globe
    # Set land to NaN
    lon_grid,lat_grid = np.meshgrid(ds.lon, ds.lat)
    globe_land_mask = globe.is_land(lat_grid,lon_grid)
    globe_land_mask_nd = np.tile(globe_land_mask,(ds[var].shape[0],ds[var].shape[1], 1,1))
    ds_no_land = xr.where(globe_land_mask_nd==True,np.nan,ds[var]) 
    return ds_no_land

In [12]:
def fix_coords(data):
    data = data.bounds.add_bounds("X")
    data = data.bounds.add_bounds("Y")
    data = data.bounds.add_bounds("T")
    data = xc.swap_lon_axis(data, to=(-180, 180))
    return data

def remove_land_full(ds, var="skt"):
    ds = xc.swap_lon_axis(ds, to=(-180, 180))
    from global_land_mask import globe
    # Set land to NaN
    lon_grid,lat_grid = np.meshgrid(ds.lon, ds.lat)
    globe_land_mask = globe.is_land(lat_grid,lon_grid)
    globe_land_mask_nd = np.tile(globe_land_mask,(ds[var].shape[0],ds[var].shape[1], 1,1))
    ds_no_land = xr.where(globe_land_mask_nd==True,np.nan,ds[var]) 
    return ds_no_land


def get_rolling_timeseries(data: xr.DataArray, window: int = 12*30, step: int = 12, save: bool = False, name: str = "", gradient: bool = False) -> xr.DataArray:
    """Get the rolling timeseries of a dataset (optionally calculate gradient)

    Args:
        data (xr.DataArray): Input data.
        window (int, optional): Size of the rolling window. Defaults to 12*30.
        step (int, optional): Step size for each window. Defaults to 12.
        gradient (bool, optional): Calculate gradient if True, mean if False. Defaults to False.

    Returns:
        xr.DataArray: Resulting timeseries with same shape as input.
    """
    
    # Prepare output array with the same shape as input data
    new_time_size = 1 + (len(data.time) - window) // step
    rolling_shape = list(data.shape)
    rolling_shape[data.get_axis_num('time')] = new_time_size

    # Prepare output array with the adjusted shape
    rolling = np.full(rolling_shape, np.nan)
    time_idx = xr.DataArray(np.arange(window), dims="time")

    data = data.chunk({"time": -1})
    # Loop to compute rolling statistics over time dimension
    for j, i in enumerate(range(0, len(data.time) - window, step)):
        if gradient:
            rolling[:,j] = window*xscore.linslope(time_idx, data.isel(time=slice(i, i+window)), dim='time', skipna=True).values
        else:
            rolling[:,j] = data.isel(time=slice(i, i+window)).mean(dim="time").values

    # Convert to xarray
    da = xr.DataArray(rolling, dims=data.dims, coords={**data.coords, 'time': np.arange(rolling.shape[1])})

    return da


def lagged_regression(ts_so_ssts, ts_ept_ssts, lags, models):
    reg_coeffs, rvalues, pvalues = [], [], []
    for lag in lags:
        rvalues.append(xscore.pearson_r(ts_so_ssts.shift(time=lag), ts_ept_ssts, dim="time", skipna=True))
        reg_coeffs.append(xscore.linslope(ts_so_ssts.shift(time=lag), ts_ept_ssts, dim="time", skipna=True))
        pvalues.append(xscore.pearson_r_eff_p_value(ts_so_ssts.shift(time=lag), ts_ept_ssts, dim="time", skipna=True))


    reg_coeffs = xr.Dataset({'reg': (['lags', 'model'], np.array(reg_coeffs))}, coords={'model': models, 'lags': lags})
    rvalues = xr.Dataset({'rvalues': (['lags', 'model'], np.array(rvalues))}, coords={'model': models, 'lags': lags})
    pvalues = xr.Dataset({'pvalues': (['lags', 'model'], np.array(pvalues))}, coords={'model': models, 'lags': lags})
    # Combine to xr.Datasets into one
    regs_dataset = xr.merge([rvalues, reg_coeffs, pvalues])
    return regs_dataset


def get_triangle(tos, latmin: float = -38.75, latmax: float = -1.25, lonmin: float = -178.75, lonmax: float = -71.25, RES: float = 2.5):
    DY = latmax - latmin
    DX = lonmax - lonmin 
    dx = RES*round(DX/DY)
    dy = RES

    print(f"For each latitude step of {dy} degrees, longitude step is {dx}")

    latcoords = np.arange(latmax, latmin-dy, -dy)
    loncoords = np.arange(lonmin, lonmax+dx, dx)
    lonraw = np.arange(lonmin, lonmax+dx, RES)

    ctos = tos.sel(lon=slice(lonmin, lonmax), lat=slice(latmin, latmax))
    nmodel, ntime, _, nlon = ctos.shape
    # print(ctos)

    for i, clon in enumerate(lonraw):
        j = np.where(clon == loncoords)[0]

        if i == nlon: break

        # print("j prior: ", j)
        if len(j) == 0: 
            j = jold
        else: 
            j = j[0]
             
        # print("j: ", j)
        nlats = int(len(latcoords) - j) # nlats below diag
        # print("nlats: ", nlats)
        ctos[:,:,:nlats,i] = np.full((nmodel, ntime, nlats), np.nan) 
        
        jold = j
    
    return ctos

def detrend(da):
    da = da.chunk(dict(time=-1))
    time_idx = xr.DataArray(np.arange(len(da.time), dtype="float32"), dims="time")
    slope = xscore.linslope(time_idx, da, dim="time", skipna=True)
    da = da - slope*time_idx
    return da 

def poly_detrend(da, deg=2):
    da = da.chunk(dict(time=-1))
    time_idx = xr.DataArray(np.arange(len(da.time), dtype="float32"), dims="time")

    def fit_and_remove_poly(y):
        mask = np.isfinite(y)
        if mask.sum() < deg + 1:
            return y  # not enough points
        coeffs = polyfit(time_idx[mask], y[mask], deg)
        trend = polyval(time_idx, coeffs)
        return y - trend

    da_dt = xr.apply_ufunc(
        fit_and_remove_poly,
        da,
        input_core_dims=[["time"]],
        output_core_dims=[["time"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[da.dtype],
    )
    return da_dt



# Preprocess

In [4]:
# Loading PiControl data
def load_cmip_data(fp):
    """Load CMIP data from a given file path."""
    # Open the dataset with xarray
    cmip_monthly_ssts = xr.open_dataset(fp, chunks="auto")
    # Fix coordinates and remove land
    cmip_monthly_ssts = fix_coords(remove_land_full(cmip_monthly_ssts, var="tos").to_dataset())
    # Detrend
    # cmip_monthly_ssts["tos"] = detrend(cmip_monthly_ssts["tos"])
    cmip_monthly_ssts["tos"] = poly_detrend(cmip_monthly_ssts["tos"], deg=2)

    # Get the EPT SSTs 
    cmip_ept_ssts = fix_coords(get_triangle(cmip_monthly_ssts.tos.copy(deep=True)).to_dataset()).spatial.average("tos").temporal.departures("tos", "month")["tos"]
    # Get the SO SSTs
    cmip_so_ssts = cmip_monthly_ssts.sel(lon=slice(-180, -75), lat=slice(-70, -50)).spatial.average("tos").temporal.departures("tos", "month")["tos"]

    return cmip_so_ssts, cmip_ept_ssts

so6_pi, ept6_pi = load_cmip_data("/home/espinosa10/tropical_pacific_clouds/data/piControl/tos_mon_1850-2100_CMIP5_piControl.nc")
so6_hi, ept6_hi = load_cmip_data("/home/espinosa10/tropical_pacific_clouds/data/historical/tos_mon_1850-2100_CMIP6_historical.nc")
so6_hi = so6_hi.load()
ept6_hi = ept6_hi.load()

For each latitude step of 2.5 degrees, longitude step is 7.5
For each latitude step of 2.5 degrees, longitude step is 7.5


In [5]:
def calculate_coupling_depending_on_time(so_ssts, ept_ssts, models):
    min_year, max_year = 1, 30
    months = np.arange(1, 12)
    years = np.arange(min_year*12, (max_year+1)*12, 12)

    time = [*months, *years]
    # time = [60]
    reg_coeffs, rvalues, pvalues, variance_so, variance_ept = [], [], [], [], []
    for year in time:
        ts_so_ssts = get_rolling_timeseries(so_ssts, window=year, step=12, gradient=False)
        ts_ept_ssts = get_rolling_timeseries(ept_ssts, window=year, step=12, gradient=False)

        variance_ept.append(ts_ept_ssts.std(dim="time"))
        variance_so.append(ts_so_ssts.std(dim="time"))
        rvalues.append(xscore.pearson_r(ts_so_ssts, ts_ept_ssts, dim="time", skipna=True))
        pvalues.append(xscore.pearson_r_eff_p_value(ts_so_ssts, ts_ept_ssts, dim="time", skipna=True))
        reg_coeffs.append(xscore.linslope(ts_so_ssts, ts_ept_ssts, dim="time", skipna=True))

    reg_coeffs = xr.Dataset({'reg': (['years', 'model'], np.array(reg_coeffs))}, coords={'model': models, 'years': time})
    rvalues = xr.Dataset({'rvalues': (['years', 'model'], np.array(rvalues))}, coords={'model': models, 'years': time})
    pvalues = xr.Dataset({'pvalues': (['years', 'model'], np.array(pvalues))}, coords={'model': models, 'years': time})
    variance_so = xr.Dataset({'variance_so': (['years', 'model'], np.array(variance_so))}, coords={'model': models, 'years': time})
    variance_ept = xr.Dataset({'variance_ept': (['years', 'model'], np.array(variance_ept))}, coords={'model': models, 'years': time})

    # Combine to xr.Datasets into one
    regs_dataset = xr.merge([rvalues, reg_coeffs, pvalues, variance_so, variance_ept])
    return regs_dataset

# Historical Coupling
if not os.path.exists("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling_historical_poly-detrended.nc"):
    so_ept_coupling_hi = calculate_coupling_depending_on_time(so6_hi, ept6_hi, models=ept6_hi.model)
    so_ept_coupling_hi.to_netcdf("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling_historical_poly-detrended.nc")
else: 
    so_ept_coupling_hi = xr.open_dataset("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling_historical_poly-detrended.nc")

# PiControl Coupling
if not os.path.exists("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling.nc"):
    so_ept_coupling_pi = calculate_coupling_depending_on_time(so6_pi, ept6_pi, models=ept6_pi.model)
    so_ept_coupling_pi.to_netcdf("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling.nc")
else: 
    so_ept_coupling_pi = xr.open_dataset("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling.nc")

# Figure SX: Historical vs. PiControl

In [6]:
so_ept_hi = xr.open_dataset("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling_historical.nc")
so_ept_pi = xr.open_dataset("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling.nc")

In [ ]:
# SWCF EPSA
swcf_cmip6_epsa = xr.open_dataarray("/home/espinosa10/tropical_pacific_clouds/data/piControl/swcf_east_sa_cmip6_v3.nc") 
swcf_cmip5_epsa = xr.open_dataarray("/home/espinosa10/tropical_pacific_clouds/data/piControl/swcf_east_sa_cmip5_v3.nc") 
swcf_cmip_epsa = xr.concat([swcf_cmip6_epsa, swcf_cmip5_epsa], dim="model")

obs_swcf = xr.open_dataset(f"/home/espinosa10/SO-EP-teleconnection/Data/obs/swcf-toa-obs-epsa.nc").swcf.values

cre = xc.open_dataset("/home/espinosa10/tropical_pacific_clouds/data/historical/swcre_cmip6_monthly_1850-2000.nc", chunks={"time": -1})
cre_anoms = xc.open_dataset("/home/espinosa10/tropical_pacific_clouds/data/historical/swcre_cmip6_anoms_monthly_1850-2000.nc", chunks={"time": -1})
tos_cmip6 = xc.open_dataset("/home/espinosa10/tropical_pacific_clouds/data/historical/tos_mon_1850-2100_CMIP6_historical.nc", chunks={"time": -1})
tos_anoms_cmip6 = tos_cmip6.temporal.departures("tos", "month").chunk({"time": -1})

tos_cmip6["tos"] = detrend(tos_cmip6.tos)
tos_anoms_cmip6["tos"] = detrend(tos_anoms_cmip6.tos)
cre_anoms["swcre"] = detrend(cre_anoms.swcre_anoms)
cre["swcre"] = detrend(cre.swcre)

cf_cmip6 = calculate_cf(tos_anoms_cmip6, cre_anoms, save=False, save_name="cmip6")
cf_cmip6 = fix_coords_no_time(cf_cmip6)
cf_cmip6_epsa = cf_cmip6.sel(lon=slice(-105, -70), lat=slice(-40, -10))
cf_cmip6_epsa = cf_cmip6_epsa.spatial.average("swcf")["swcf"]
cf_cmip6_epsa

2025-08-05 08:08:14,741 [WARNING]: bounds.py(_create_bounds:398) >> The 'lat' coordinate variable is missing a 'units' attribute. Assuming 'units' is 'degrees_north'.


<xarray.DataArray 'swcf' (model: 50)>
dask.array<truediv, shape=(50,), dtype=float64, chunksize=(50,), chunktype=numpy.ndarray>
Coordinates:
  * model    (model) object 'CESM2-WACCM' 'IPSL-CM5A2-INCA' ... 'BCC-ESM1'

In [22]:
def fix_coords_no_swap(data):
    data = data.bounds.add_bounds("X")
    data = data.bounds.add_bounds("Y")
    data = data.bounds.add_bounds("T")
    return data

DATA_ROOT = "/home/espinosa10/SO-EP-teleconnection/Data"
# Calculate Southern ITCZ from CMIP
if not os.path.exists(os.path.join(DATA_ROOT, "cmip_pr_south_east_clim")):
    pr_cmip6 = xr.open_dataset("/home/espinosa10/tropical_pacific_clouds/data/piControl/pr_mon_1850-2100_CMIP6_piControl.nc")
    pr_cmip5 = xr.open_dataset("/home/espinosa10/tropical_pacific_clouds/data/piControl/pr_mon_1850-2100_CMIP5_piControl.nc")
    pr_cmip =  xr.concat([pr_cmip6, pr_cmip5], dim="model")
    pr_cmip = fix_coords_no_swap(pr_cmip)
    pr_south_east = pr_cmip.sel(lon=slice(230, 280), lat=slice(-20, 0)).spatial.average("pr")["pr"]
    cmip_precip_south_east_clim = pr_south_east.mean("time")*86400
    cmip_precip_south_east_clim.to_netcdf(os.path.join(DATA_ROOT, "cmip_pr_south_east_clim.nc"))
else: 
    cmip_precip_south_east_clim = xr.open_dataset(os.path.join(DATA_ROOT, "cmip_pr_south_east_clim.nc"))["pr"]

cmip_precip_south_east_clim

<xarray.DataArray 'pr' (model: 100)>
array([1.88052879e+00, 1.67439962e+00, 2.02557613e+00, 1.73055361e-03,
       2.00372076e+00, 2.11481841e+00, 6.88531523e-01, 1.52321397e+00,
       1.37531860e+00, 7.79833872e-01, 1.18365595e+00, 2.47875980e+00,
       1.21882217e+00, 1.24227043e+00, 1.66170821e+00, 1.41725074e+00,
       1.50185293e+00, 1.20323509e+00, 3.32023363e+00, 3.32574890e+00,
       2.85652621e+00, 3.33000579e+00, 3.30003566e+00, 2.35062949e+00,
       1.57730665e+00, 2.26970609e+00, 7.06878168e-01, 2.28712825e+00,
       2.28592367e+00, 2.25981077e+00, 1.14164197e+00, 1.94193004e+00,
       1.88786563e+00, 1.88417933e+00, 1.60623129e+00, 1.61508785e+00,
       1.44963302e+00, 1.64434610e+00, 1.74181428e+00, 1.90648019e+00,
       1.01740981e+00, 2.17900442e+00, 1.91532234e+00, 1.09828782e+00,
       1.01836023e+00, 2.52077029e+00, 1.58960127e+00, 1.62506896e+00,
       1.13342983e+00, 1.34555040e+00, 1.06355728e+00, 1.67996163e+00,
       1.14276660e+00, 1.69450952e+00, 9.68291686e-01, 7.53224108e-01,
       6.61878373e-01, 2.29147852e+00, 2.06933167e+00, 2.93408961e+00,
       2.04716461e+00, 2.27012509e+00, 2.38429754e+00, 2.10607344e+00,
       1.92791036e+00, 1.93579036e+00, 1.75971958e+00, 1.79628447e+00,
       2.13006598e+00, 2.03297547e+00, 2.23806451e+00, 2.70769519e+00,
       2.70769519e+00, 1.29315740e+00, 1.60091275e+00, 1.15884922e+00,
       1.22023511e+00, 3.81940179e+00, 2.76729222e+00, 2.74251409e+00,
       3.86120003e+00, 9.27908123e-01, 1.56539772e+00, 2.98167441e+00,
       1.17389761e+00, 8.38775815e-01, 1.03419489e+00, 1.34667642e+00,
       2.20693315e+00, 1.63756193e+00, 1.24767076e+00, 1.27242763e+00,
       1.58463768e+00, 1.67705524e+00, 1.38525172e+00, 1.17148069e+00,
       1.19593570e+00, 1.78146731e+00, 2.42299244e+00, 2.21204027e+00])
Coordinates:
  * model    (model) object 'E3SM-1-1' 'E3SM-1-0' ... 'MRI-CGCM3' 'CNRM-CM5-2'

In [17]:
def create_panel_a(ax, x6, y6, x5: list = [], y5: list = [], vline=0) -> None:
    # Set the default color cycle
    colors = ['black','maroon','#7570b3']
    mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=colors)

    CMIP6_label = "CMIP6"
    
    if len(x5) != 0: 
        slope, intercept, r, p, _ = linregress(x5,y5)
        r, p = np.around(r, 3), np.around(p, 3)
        CMIP5_label = f"r={r}"
        ax.scatter(x5, y5, s=100, alpha=.5, edgecolor="black", color=colors[0], label=CMIP5_label)

        slope, intercept, r, p, _ = linregress(x6,y6)
        r, p = np.around(r, 3), np.around(p, 3)
        CMIP6_label = f"r={r}"
    
    ax.scatter(x6, y6, s=100, alpha=.5, edgecolor="black", color=colors[0], label=CMIP6_label)

    # Multimodel mean
    mmm = np.mean([*x5, *x6])


    x = np.concatenate((x6, x5))
    y = np.concatenate((y6, y5))
    # Sort the lists together based on list1
    x, y  = zip(*sorted(zip(x, y)))

    slope, intercept, r, p, _ = linregress(x,y)
    r, p = np.around(r, 3), np.around(p, 3)
    if p < .05: 
        sig = '*'
    else: 
        sig = ''
    print(r, p)

    x = np.array(sorted(x))
    ax.plot(x, x*slope+intercept, color="black")
    # ax.legend(loc="upper left", fontsize=fontsize)
    ax.set_title(f"r={r}{sig}", loc="right", fontweight="bold", fontsize=16)
    
    
    # Define border size - important for emphasizing relationships
    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)
    borderx, bordery = .25*abs(np.max(x)), .25*abs(np.max(y))
    ax.set_xlim(xmin-borderx, xmax+borderx)
    ax.set_ylim(ymin-bordery, ymax+bordery)

    # Vertical line for avg and std of obs
    ax.vlines(np.mean(vline), ymin=ymin-bordery, ymax=ymax+bordery, color="orange")
    ax.fill_between(x=[np.mean(vline) - np.std(vline), np.mean(vline) + np.std(vline)], y1=ymin-bordery, y2=ymax+bordery,color="orange", alpha=.25)

    # Add Model Vertical Lines
    ax.vlines(mmm, ymin=ymin-bordery, ymax=ymax+bordery, color=colors[0], label='_nolegend_', alpha=.5, linewidth=2, zorder=0) 
    # ax.fill_between(x=[mmm - np.std(x), mmm + np.std(x)], y1=-1, y2=2.5, color="black", alpha=.25)
    ax.fill_between(x=[mmm - np.std(x), mmm + np.std(x)], y1=np.min(y)*5-50, y2=np.max(y)*5+50, color="black", alpha=.25)

    # Add horizontal sigma lines
    sig_mmm = mmm*slope+intercept
    sig_obs = np.mean(vline)*slope+intercept
    hline_width = 3
    ax.hlines(y=sig_mmm, xmin=xmin-borderx, xmax=mmm, color="black", linewidth=hline_width)
    # ax.hlines(y=sig_obs, xmin=xmin-borderx, xmax=np.mean(vline), color="navy", linewidth=hline_width)
    ax.hlines(y=sig_obs, xmin=xmin-borderx, xmax=np.mean(vline), color="orange", linewidth=hline_width)

    ax.text(s=f"{np.around(sig_mmm, 2)}", x=xmin-borderx*.95, y=sig_mmm + sig_mmm*.025, color="black", fontsize=12, fontweight="bold")
    ax.text(s=f"{np.around(sig_obs, 2)}", x=xmin-borderx*.95, y=sig_obs + sig_mmm*.025, color="orange", fontsize=12, fontweight="bold")
    # ax.text(s=f"{np.around(sig_obs, 2)}", x=xmin-borderx*.95, y=sig_obs + sig_mmm*.025, color="navy", fontsize=12, fontweight="bold")

def get_shared_models(ds1: xr.Dataset, ds2: xr.Dataset) -> Tuple[xr.Dataset, xr.Dataset]:
    shared_models = list(set(ds1.model.values).intersection(set(ds2.model.values)))
    return ds1.sel(model=shared_models), ds2.sel(model=shared_models)

In [ ]:
title_fontsize = 16
# Create figure and gridspec
fig = plt.figure(figsize=(12,12))
gs = fig.add_gridspec(2, 2, wspace=0.25, hspace=0.25)  # 2 rows, 2 columns

# Create scatter plots (top row)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

######## 1. dEPT/dSO Hist vs dEPT/dSO PiControl ############
x, y = get_shared_models(so_ept_pi["reg"].sel(years=12*5), so_ept_hi["reg"].sel(years=12*5))
create_panel_a(ax1, x.values, y.values, vline=[])
ax1.set_ylabel(r"$\frac{d EPT_{SST}}{dSO_{SST}}_{Historical}$", fontsize=18)
ax1.set_xlabel(r"$\frac{d EPT_{SST}}{dSO_{SST}}_{PiControl}$", fontsize=18)
ax1.plot(np.arange(-.5, 2.5, .1), np.arange(-.5, 2.5, .1), color="black", linestyle="--", linewidth=1.5, label="_nolegend_")
ax1.set_title(r"A", loc="left", fontweight="bold", fontsize=title_fontsize)
ax1.set_ylim(-.5, 2.5)
ax1.set_xlim(-.5, 2.5)
ax1.grid()

######## 2. SWCF Hist vs SWCF PiControl ############
x, y = get_shared_models(swcf_cmip_epsa, cf_cmip6_epsa)
create_panel_a(ax2, x.values, y.values, vline=[])
ax2.set_ylabel(r"$\lambda_{SW}$ EPSA ($Wm^{-2}K^{-1}$) Historical", fontsize=12)
ax2.set_xlabel(r"$\lambda_{SW}$ EPSA ($Wm^{-2}K^{-1}$) PiControl", fontsize=12)
ax2.plot(np.arange(-5, 16, 1), np.arange(-5, 16, 1), color="black", linestyle="--", linewidth=1.5, label="_nolegend_")
ax2.set_title(r"B", loc="left", fontweight="bold", fontsize=title_fontsize)
ax2.set_xlim(-2, 14)
ax2.set_ylim(-2, 14)
ax2.grid()

######## 3. dEPT/dSO Hist vs SWCF Hist ############
x, y = get_shared_models(cf_cmip6_epsa, so_ept_hi["reg"].sel(years=12*5))
create_panel_a(ax3, x.values, y.values, vline=[])
ax3.set_ylabel(r"$\frac{d EPT_{SST}}{dSO_{SST}}_{Historical}$", fontsize=18)
ax3.set_xlabel(r"$\lambda_{SW}$ EPSA ($Wm^{-2}K^{-1}$) Historical", fontsize=12)
ax3.set_title(r"C", loc="left", fontweight="bold", fontsize=title_fontsize)
ax3.set_xlim(-2, 14)
ax3.set_ylim(-.5, 2.5)
ax3.grid()

######## 4. dEPT/dSO Hist vs Clim Precip PiControl ############
x, y = get_shared_models(cf_cmip6_epsa, cmip_precip_south_east_clim)
create_panel_a(ax4, x.values, y.values, vline=[])
ax4.set_ylabel(r"Precip Southern ITCZ ($mm/day$)", fontsize=12)
ax4.set_xlabel(r"$\lambda_{SW}$ EPSA ($Wm^{-2}K^{-1}$) Historical", fontsize=12)
ax4.set_title(r"D", loc="left", fontweight="bold", fontsize=title_fontsize)
ax4.set_xlim(-2, 14)
ax4.set_ylim(0, 3.5)
ax4.grid()

#########################################################################
plt.savefig("/home/espinosa10/SO-EP-teleconnection/Manuscript_Figures/SI/Figure_SX_hi_vs_pi.png", dpi=300, bbox_inches="tight")
plt.savefig("/home/espinosa10/SO-EP-teleconnection/Manuscript_Figures/SI/Figure_SX_hi_vs_pi.pdf", dpi=300, bbox_inches="tight")


0.799 0.0
